In [10]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

img_129 = Image.open("train/129.jpg")
s1 = img_129.size[0] * img_129.size[1]

df_train = pd.read_csv("train_data.csv")
label_counts = df_train['Label'].value_counts()
s2 = round(label_counts.min() / label_counts.max(), 6)

subtasks = pd.DataFrame({
    "subtaskID": [1, 2],
    "datapointID": [0, 0],
    "answer": [s1, s2]
})

IMG_SIZE = 128
X, y = [], []

for _, row in df_train.iterrows():
    img_path = f"train/{row['CodeID']}.jpg"
    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        X.append(img)
        y.append(row['Label'])

X = np.array(X) / 255.0
y = to_categorical(y)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(min_delta = 0.001, patience=20, restore_best_weights=True)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=75, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])

test_dir = "test"
test_images = []
test_ids = []

for fname in sorted(os.listdir(test_dir), key=lambda x: int(x.split('.')[0])):
    if fname.endswith(".jpg"):
        img_id = int(fname.split('.')[0])
        img_path = os.path.join(test_dir, fname)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            test_images.append(img)
            test_ids.append(img_id)

X_test = np.array(test_images) / 255.0
preds = model.predict(X_test)
pred_labels = np.argmax(preds, axis=1)

df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": test_ids,
    "answer": pred_labels
})

submission = pd.concat([subtasks, df_preds])
submission = submission.sort_values(by=["subtaskID", "datapointID"])
submission.to_csv("submission.csv", index=False)
submission

Epoch 1/75


C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 254ms/step - accuracy: 0.5273 - loss: 0.8255 - val_accuracy: 0.4667 - val_loss: 0.6881
Epoch 2/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 213ms/step - accuracy: 0.5961 - loss: 0.6823 - val_accuracy: 0.6667 - val_loss: 0.6544
Epoch 3/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 221ms/step - accuracy: 0.7795 - loss: 0.6280 - val_accuracy: 0.6667 - val_loss: 0.6110
Epoch 4/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step - accuracy: 0.7818 - loss: 0.5056 - val_accuracy: 0.6889 - val_loss: 0.6393
Epoch 5/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 219ms/step - accuracy: 0.7341 - loss: 0.5118 - val_accuracy: 0.6222 - val_loss: 0.5918
Epoch 6/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 253ms/step - accuracy: 0.8035 - loss: 0.4470 - val_accuracy: 0.5778 - val_loss: 0.5892
Epoch 7/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 214ms/step - accuracy: 0.8249 - loss: 0.4069 - val_accuracy: 0.5778 - val_loss: 0.6159
Epoch 8/75
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 214ms/step - accuracy: 0.8850 - loss: 0.3430 - val_accuracy: 0.7111 - val_loss: 0.7516
Epo

,subtaskID,datapointID,answer
0,1,0,353808.000000
1,2,0,0.715385
0,3,0,0.000000
1,3,1,0.000000
2,3,2,0.000000
...,...,...,...
112,3,5047,0.000000
113,3,5048,0.000000
114,3,5049,0.000000
115,3,5050,1.000000
